# Results for model: mistralai/mixtral-8x7b-instruct-v0.1

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

# 1. Load 'train.parquet' using Polars.
df = pl.read_parquet("./jane_street_data/train.parquet")

# 2. Feature Engineering:
# Calculate a global TOP_QUANTILE (top 15%) of 'feature_00'
# relative to 'responder_6' across rolling batches of 'date_id'.

batch_size = int(df.height() * 0.15)

def top_quantile_feature(batch):
    global TOP_QUANTILE
    return (
        batch["feature_00"]
        .sort_by("feature_00", reverse=True)
        .roll_windows(batch_size)
        .apply(lambda s: s.item().quantile(0.95) if s.len() >= batch_size else None)
        .alias("top_feature_00")
    )

TQ = pl.lit(None).alias("TOP_QUANTILE")
df = df.with_column(TQ)

rolled_series = df.groupby_rolling("date_id", period=batch_size, maintain_order=True)
df = rolled_series.apply(top_quantile_feature).with_columns(pl.col("TOP_QUANTILE").fill_null(TQ))

# 3. Train an XGBoost Regressor on the target 'responder_6'.

TQ_col = pl.col("TOP_QUANTILE")
X = pl.concat([
    df
    .select(pl.exclude("responder_6", "date_id", "TOP_QUANTILE"))
    .cast(TQ_col.dtype)
    .rename({TQ_col.name: "top_feature_00"})
], horizontally=True)

y = df.select("responder_6")

# Prepare Dataset
def prepare_dataset(dataset):
    data = np.array(dataset)
    X = data[:, :-1]
    y = data[:, -1]

    return xgb.DMatrix(X, label=y)

X_train = prepare_dataset(X.sort("date_id"))
y_train = prepare_dataset(y.sort("date_id"))

# Train Model
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
}

xg_reg = xgb.train(params, X_train, num_boost_round=1000, evals=[(y_train, 'train')], early_stopping_rounds=50)